# Analyze ONNX for GPT

Analyze one NeuroGolf ONNX file and generate a compact text package that can be pasted into GPT for optimization ideas.

In [89]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from neurogolf_score import score_model_file
from neurogolf_onnx_analysis import (
    compare_score_rows,
    compact_model_markdown,
    gpt_analysis_prompt,
    run_model_on_examples,
    summarize_model,
    summarize_task,
)

DATA_DIR = repo_root / "neurogolf-2026"
WORK_DIR = repo_root / "outputs" / "gpt_onnx_analysis"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TASK_ID = "task001"
ONNX_PATH = repo_root / "solution" / "6410.88" / f"{TASK_ID}.onnx"
CANDIDATE_ONNX_DIR = repo_root / "outputs" / "gpt_workbench" / TASK_ID
CANDIDATE_ONNX_PATHS = sorted(CANDIDATE_ONNX_DIR.glob("*.onnx"))

TASK_ID, ONNX_PATH, CANDIDATE_ONNX_DIR, CANDIDATE_ONNX_PATHS


('task001',
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6410.88/task001.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task001'),
 [])

## Task Summary

In [90]:
task_summary = summarize_task(TASK_ID, DATA_DIR)

print(f"task    : {task_summary['task']}")
print(f"train   : {task_summary['num_train']}")
print(f"test    : {task_summary['num_test']}")
print(f"arc-gen : {task_summary['num_arc_gen']}")
print(f"colors  : {task_summary['color_counts']}")

try:
    import pandas as pd

    display(pd.DataFrame(task_summary["examples"]))
except ImportError:
    task_summary["examples"]

task    : task001
train   : 5
test    : 1
arc-gen : 262
colors  : {0: 15578, 1: 824, 2: 1106, 3: 608, 4: 742, 5: 1054, 6: 1170, 7: 1336, 8: 812, 9: 890}


,subset,index,input_shape,output_shape,input_cells,output_cells
0,train,0,3x3,9x9,9,81
1,train,1,3x3,9x9,9,81
2,train,2,3x3,9x9,9,81
3,train,3,3x3,9x9,9,81
4,train,4,3x3,9x9,9,81
...,...,...,...,...,...,...
263,arc-gen,257,3x3,9x9,9,81
264,arc-gen,258,3x3,9x9,9,81
265,arc-gen,259,3x3,9x9,9,81
266,arc-gen,260,3x3,9x9,9,81


## Score

In [91]:
base_score_row = score_model_file(ONNX_PATH, DATA_DIR)
candidate_score_rows = []

for candidate_path in CANDIDATE_ONNX_PATHS:
    row = score_model_file(candidate_path, DATA_DIR)
    row["candidate_file"] = candidate_path.name
    row["candidate_path"] = str(candidate_path)
    row.update(compare_score_rows(base_score_row, row))
    candidate_score_rows.append(row)

# Keep this alias for the GPT prompt generation cells below.
score_row = base_score_row

print("Base/reference ONNX:")
display(base_score_row)

print(f"Candidate ONNX files: {len(candidate_score_rows)}")
try:
    import pandas as pd

    if candidate_score_rows:
        score_cols = [
            "candidate_file", "status", "score", "score_delta", "cost", "cost_delta",
            "memory", "memory_delta", "params", "params_delta", "filesize", "filesize_delta", "error"
        ]
        display(pd.DataFrame(candidate_score_rows)[score_cols].sort_values("score", ascending=False).reset_index(drop=True))
    else:
        print(f"No candidate ONNX files found in {CANDIDATE_ONNX_DIR}")
except ImportError:
    candidate_score_rows


Base/reference ONNX:


{'task': 'task001',
 'candidate': '6410.88',
 'file': '/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6410.88/task001.onnx',
 'filesize': 872,
 'status': 'ok',
 'memory': 2459,
 'params': 23,
 'cost': 2482.0,
 'score': 17.18318003423545,
 'error': ''}

Candidate ONNX files: 0
No candidate ONNX files found in /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task001


## Candidate Comparison

In [92]:
base_validation = run_model_on_examples(ONNX_PATH, TASK_ID, DATA_DIR)
candidate_validation_rows = []
candidate_validations = {}

for candidate_path in CANDIDATE_ONNX_PATHS:
    report = run_model_on_examples(candidate_path, TASK_ID, DATA_DIR)
    candidate_validations[candidate_path.name] = report
    counts = report.get("counts", {})
    candidate_validation_rows.append({
        "candidate_file": candidate_path.name,
        "validation_status": report.get("status"),
        "pass": counts.get("pass", 0),
        "fail": counts.get("fail", 0),
        "error": counts.get("error", 0),
        "skipped_large_grid": counts.get("skipped_large_grid", 0),
        "first_failure": report.get("first_failure"),
    })

print("Base validation:")
display({k: v for k, v in base_validation.items() if k != "rows"})

print("Candidate validations:")
try:
    import pandas as pd

    validation_df = pd.DataFrame(candidate_validation_rows)
    if candidate_score_rows:
        score_df = pd.DataFrame(candidate_score_rows)
        compare_df = validation_df.merge(
            score_df[["candidate_file", "status", "score", "score_delta", "cost_delta", "memory_delta", "params_delta", "filesize_delta"]],
            on="candidate_file",
            how="left",
        )
        display(compare_df.sort_values(["validation_status", "score_delta"], ascending=[True, False]).reset_index(drop=True))
    else:
        display(validation_df)
except ImportError:
    candidate_validation_rows


Base validation:


{'status': 'ok', 'counts': {'pass': 268}, 'first_failure': None}

Candidate validations:


""


In [93]:
valid_improvements = []
for row in candidate_score_rows:
    validation = candidate_validations.get(row["candidate_file"], {})
    if validation.get("status") == "ok" and row.get("score_delta", 0) > 0:
        valid_improvements.append(row)

print(f"Valid score improvements: {len(valid_improvements)}")
try:
    import pandas as pd

    if valid_improvements:
        display(pd.DataFrame(valid_improvements).sort_values("score_delta", ascending=False).reset_index(drop=True))
except ImportError:
    valid_improvements


Valid score improvements: 0


In [94]:
failed_candidates = {
    name: report.get("first_failure")
    for name, report in candidate_validations.items()
    if report.get("first_failure")
}
failed_candidates


{}

## ONNX Structure

In [95]:
model_summary = summarize_model(ONNX_PATH)

print(f"file         : {model_summary['filename']}")
print(f"filesize     : {model_summary['filesize']}")
print(f"ir_version   : {model_summary['ir_version']}")
print(f"opset_imports: {model_summary['opset_imports']}")
print(f"inputs       : {model_summary['inputs']}")
print(f"outputs      : {model_summary['outputs']}")
print(f"nodes        : {model_summary['num_nodes']}")
print(f"initializers : {model_summary['num_initializers']}")
print(f"op_counts    : {model_summary['op_counts']}")

file         : task001.onnx
filesize     : 872
ir_version   : 8
opset_imports: [{'domain': '', 'version': 9}]
inputs       : [{'name': 'input', 'elem_type': 'FLOAT', 'shape': [1, 10, 30, 30]}]
outputs      : [{'name': 'output', 'elem_type': 'FLOAT16', 'shape': [1, 10, 30, 30]}]
nodes        : 12
initializers : 3
op_counts    : {'Cast': 2, 'Gather': 2, 'Or': 1, 'Pad': 2, 'ReduceMax': 1, 'Slice': 2, 'Tile': 1, 'Where': 1}


## Initializers

In [96]:
try:
    import pandas as pd

    display(pd.DataFrame(model_summary["initializers"]))
except ImportError:
    model_summary["initializers"]

,name,dtype,dims,numel,bytes,sample
0,r,INT64,[4],4,32,"[1, 1, 3, 3]"
1,i,INT64,[9],9,72,"[0, 0, 0, 1, 1, 1, 2, 2, 2]"
2,bo,FLOAT16,"[1, 10, 1, 1]",10,20,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


## Nodes

In [97]:
try:
    import pandas as pd

    nodes_df = pd.DataFrame(model_summary["nodes"])
    display(nodes_df[["index", "op_type", "name", "inputs", "outputs", "attributes"]])
except ImportError:
    model_summary["nodes"][:20]

,index,op_type,name,inputs,outputs,attributes
0,0,Slice,a,[input],[b0],"{'axes': [1, 2, 3], 'ends': [1, 3, 3], 'starts..."
1,1,Cast,b,[b0],[b],{'to': 9}
2,2,Tile,c,"[b, r]",[t],{}
3,3,Gather,d,"[b, i]",[g0],{'axis': 2}
4,4,Gather,e,"[g0, i]",[g],{'axis': 3}
5,5,Or,f,"[t, g]",[m],{}
6,6,Slice,g,[input],[x],"{'axes': [1, 2, 3], 'ends': [10, 3, 3], 'start..."
7,7,Cast,h,[x],[h],{'to': 10}
8,8,ReduceMax,j,[h],[cf],"{'axes': [2, 3], 'keepdims': 1}"
9,9,Pad,k,[cf],[co],"{'mode': 'constant', 'pads': [0, 1, 0, 0, 0, 0..."


## GPT Package

In [98]:
model_markdown = compact_model_markdown(model_summary, score_row=score_row, max_nodes=160)
prompt = gpt_analysis_prompt(task_summary, model_markdown)

prompt_path = WORK_DIR / f"{TASK_ID}_gpt_analysis_prompt.md"
prompt_path.write_text(prompt)

print(f"Saved GPT prompt: {prompt_path}")
print(f"Prompt characters: {len(prompt)}")

Saved GPT prompt: /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_onnx_analysis/task001_gpt_analysis_prompt.md
Prompt characters: 34775


In [99]:
print(prompt)

You are helping optimize a NeuroGolf ONNX model.

Rules:
- Input tensor: float32 [1, 10, 30, 30], name 'input'.
- Output tensor: float32 [1, 10, 30, 30], name 'output'.
- Static tensor shapes only.
- One input and one output only.
- Banned ops: Loop, Scan, NonZero, Unique, Script, Function, Compress.
- Optimize score by reducing memory + params.
- Assume public examples pass unless told otherwise.

Requested output:
1. Summarize what the current ONNX appears to do.
2. Identify redundant or expensive parts.
3. Propose safe rewrite candidates ranked by risk and likely score gain.
4. Do not write code yet.

Task summary:
- task: task001
- train examples: 5
- test examples: 1
- arc-gen examples: 262
- color counts: {0: 15578, 1: 824, 2: 1106, 3: 608, 4: 742, 5: 1054, 6: 1170, 7: 1336, 8: 812, 9: 890}
- example shapes: [{'subset': 'train', 'index': 0, 'input_shape': '3x3', 'output_shape': '9x9', 'input_cells': 9, 'output_cells': 81}, {'subset': 'train', 'index': 1, 'input_shape': '3x3', 'ou

Suggested workflow: paste the generated prompt into GPT first and ask for analysis only. After choosing a candidate rewrite, ask GPT to produce a `build_taskXXX.py` script that creates a new ONNX.